# Don Bossing — Free ComfyUI (Kaggle P100), stable tunnel

Runs ComfyUI + CogVideoX on a **free Kaggle GPU** (P100 16GB, ~30 GPU-hrs/week) and exposes it through a **stable public tunnel** so your local `server.js` can drive it. Setup once, then just re-run when the session expires.

Steps:
0. Run the first code cell (GPU check) and confirm `Tesla P100` (or T4) appears before the long download.
1. Set this notebook's accelerator to **GPU** (Settings → Accelerator → GPU P100).
2. (Stable URL) Create a **free ngrok account**, copy your authtoken + pick a subdomain (e.g. `donbossing-video`). In Kaggle: *Add-ons → Secrets* and add `NGROK_AUTHTOKEN` and `NGROK_SUBDOMAIN`. (No-signup alternative: pinggy.io — URL may change each restart.)
3. Run the cells top to bottom.
4. Copy the printed `https://<subdomain>.ngrok-free.dev` URL into `video.config.json` → `comfyUrl` on your PC (set once).
5. On your PC run `npm run local`, open http://localhost:3000, use Section 6.

When the Kaggle session expires, just re-run — the ngrok subdomain is the same, so no config change needed.

In [ ]:
# --- 0. Verify GPU + environment (run FIRST, before the long download) ---
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU DETECTED - enable GPU P100 in Settings'
import sys, os
print('Python', sys.version.split()[0])
print('cwd', os.getcwd())


In [ ]:
# --- 1. Install ComfyUI + wrappers ---
!git clone https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI
!pip install -q -r requirements.txt
!git clone https://github.com/kijai/ComfyUI-CogVideoXWrapper custom_nodes/ComfyUI-CogVideoXWrapper
!git clone https://github.com/kijai/ComfyUI-HunyuanVideoWrapper custom_nodes/ComfyUI-HunyuanVideoWrapper
!git clone https://github.com/comfyanonymous/ComfyUI-VideoHelperSuite custom_nodes/ComfyUI-VideoHelperSuite
!pip install -q -r custom_nodes/ComfyUI-CogVideoXWrapper/requirements.txt 2>/dev/null
!pip install -q -r custom_nodes/ComfyUI-HunyuanVideoWrapper/requirements.txt 2>/dev/null
print('comfyui + wrappers installed')

In [ ]:
# --- 2. Download model weights (only CogVideoX fits the free 16GB P100) ---
!mkdir -p models/CogVideoX models/text_encoders
!huggingface-cli download THUDM/CogVideoX-5b-I2V --local-dir models/CogVideoX --local-dir-use-symlinks False
!huggingface-cli download comfyanonymous/flux_text_encoders t5xxl_fp8_e4m3fn.safetensors --local-dir models/text_encoders --local-dir-use-symlinks False
print('weights ready')

In [ ]:
# --- 3. Launch ComfyUI in the background ---
import subprocess, time
log = open('comfy.log','w')
proc = subprocess.Popen(['python','main.py','--listen','0.0.0.0','--port','8188','--disable-auto-launch'],
                        stdout=log, stderr=subprocess.STDOUT)
print('ComfyUI launching (pid', proc.pid, ')...')
time.sleep(25)
print(open('comfy.log').read()[-1500:])

In [ ]:
# --- 4. Stable tunnel: ngrok reserved subdomain (recommended) ---
import os, subprocess, re
token = os.environ.get('NGROK_AUTHTOKEN','')
sub = os.environ.get('NGROK_SUBDOMAIN','')
if token and sub:
    print('BRANCH: ngrok (stable subdomain)')
    # Install the v3 agent (tgz). v2 (zip) lacks --domain and can't use *.ngrok-free.dev.
    !pkill -9 ngrok 2>/dev/null; true
    !rm -f ngrok ngrok-stable-linux-amd64.zip ngrok3.tgz
    !curl -L -s -o ngrok3.tgz https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
    !tar xzf ngrok3.tgz
    !./ngrok --version
    !./ngrok config add-authtoken {token}
    domain = sub if '.' in sub else (sub + '.ngrok-free.dev')
    tun = subprocess.Popen(['./ngrok','http','--domain='+domain,'8188'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = 'https://'+domain
    print('=== STABLE TUNNEL URL (paste once into video.config.json comfyUrl) ===')
    print(url)
    print('=====================================================================')
else:
    # No-signup fallback: pinggy.io (URL may change each restart)
    print('BRANCH: pinggy fallback (NGROK secrets not set)')
    tun = subprocess.Popen(['ssh','-p','443','-o','StrictHostKeyChecking=accept-new','-R0:localhost:8188','a.pinggy.io'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in tun.stdout:
        print(line.rstrip())
        m = re.search(r'(https://[a-z0-9.\-]+\.(?:pinggy\.link|tcp\.ngrok\.io))', line)
        if m:
            print('\n=== TUNNEL URL (paste into video.config.json comfyUrl) ===')
            print(m.group(1))
            print('=========================================================')
            break

In [ ]:
# --- 5. Keep-alive: poll ComfyUI so the session stays busy ---
import urllib.request, time, json
while True:
    try:
        with urllib.request.urlopen('http://localhost:8188/system_stats', timeout=5) as r:
            s = json.load(r)
            print('ComfyUI alive. GPU:', s.get('devices',[{}])[0].get('name','?'))
    except Exception as e:
        print('ComfyUI check:', e)
    time.sleep(60)